In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(hise)
quiet_library(dplyr)
quiet_library(purrr)

ERROR: Error in library(...): there is no package called ‘dplyr’


In [2]:
if(!dir.exists("output")) {
    dir.create("output")
}

### Retrieve sample information

In [3]:
sample_meta_file_uuid = '2da66a1a-17cc-498b-9129-6858cf639caf'

In [4]:
sample_meta_file <- cacheFiles(list(sample_meta_file_uuid))

[1] "downloading fileID 2da66a1a-17cc-498b-9129-6858cf639caf"


In [5]:
sample_meta_file <- paste0("~/", sample_meta_file)

In [6]:
sample_meta <- read.csv(sample_meta_file)

In [7]:
nrow(sample_meta)

[1] 108

### Retrieve panel information

These files list of targets, conjugates, and clones for each feature in the flow cytometry data. These are derived from the tables in Heubeck, et al. (2021).

In [8]:
panel_uuids <- list(
    "PB1" = list("28d563bf-4b35-405d-9136-56b9f1d77f24"),
    "PM1" = list("e4658085-e1a2-40f5-9e89-a3083b1758a6"),
    "PS1" = list("d4349731-4eae-4381-8519-72816b0cb9e7"),
    "PT1" = list("a1c0beeb-b951-4628-956a-2680c0d7230d")
)

In [9]:
panel_files <- map(
    panel_uuids,
    cacheFiles
)

[1] "downloading fileID 28d563bf-4b35-405d-9136-56b9f1d77f24"
[1] "downloading fileID e4658085-e1a2-40f5-9e89-a3083b1758a6"
[1] "downloading fileID d4349731-4eae-4381-8519-72816b0cb9e7"
[1] "downloading fileID a1c0beeb-b951-4628-956a-2680c0d7230d"


In [10]:
panel_files <- map(
    panel_files,
    function(f) {
        paste0("/home/workspace/", f)
    }
)

## Find flow cytometry data

### Compensated events
fileType = FlowCytometry

There should be 4 files for each sample, one per Flow Cytometry panel; 432 in total.

In [11]:
event_desc <- getFileDescriptors(
    fileType = "FlowCytometry",
    filter = list(
        sample.sampleKitGuid = as.list(sample_meta[["sample.sampleKitGuid"]])
    )
)

In [12]:
event_desc <- fileDescToDataframe(event_desc)
nrow(event_desc)

[1] 939

Some events will be from exactly the same aliquots as the scRNA-seq data.

We can match these based on specimen IDs:

In [13]:
event_desc <- event_desc %>%
  mutate(pbmc_sample_id = sub(".+(PB[0-9]+-[0-9]+).+", "\\1", file.name)) %>%
  mutate(file_date = sub(".+(20[0-9]{2}-[0-9]{2}-[0-9]{2}).+", "\\1", file.name))

In [14]:
matched_event_desc <- event_desc %>%
  filter(pbmc_sample_id %in% sample_meta$pbmc_sample_id)
nrow(matched_event_desc)

[1] 501

Remove duplicates - in some cases, files were uploaded multiple times. We'll make sure that we take the most recent data available for each sample.

In [15]:
matched_event_desc <- matched_event_desc %>%
  arrange(desc(file_date)) %>%
  group_by(pbmc_sample_id, file.panel) %>%
  slice(1)

In [16]:
nrow(matched_event_desc)

[1] 256

In [17]:
table(matched_event_desc$file.panel)


PB1 PM1 PS1 PT1 
 64  64  64  64 

In [18]:
length(unique(matched_event_desc$sample.sampleKitGuid))

[1] 64

In some cases, the specific aliquot may differ, but data may be available from the same batch:

In [19]:
batch_event_desc <- event_desc %>%
  filter(!sample.sampleKitGuid %in% matched_event_desc$sample.sampleKitGuid) %>%
  filter(file.batchID %in% sample_meta$file.batchID) %>%
  arrange(desc(file_date)) %>%
  group_by(sample.sampleKitGuid, file.panel) %>%
  slice(1)
nrow(batch_event_desc)

[1] 72

In [20]:
table(batch_event_desc$file.panel)


PB1 PM1 PS1 PT1 
 18  18  18  18 

Some samples were run in separate batches. In these cases, we'll take the most recent file per sample kit for each file panel.

In [21]:
unmatched_event_desc <- event_desc %>%
  filter(!sample.sampleKitGuid %in% matched_event_desc$sample.sampleKitGuid) %>%
  filter(!sample.sampleKitGuid %in% batch_event_desc$sample.sampleKitGuid) %>%
  arrange(desc(file_date)) %>%
  group_by(sample.sampleKitGuid, file.panel) %>%
  slice(1)
nrow(unmatched_event_desc)

[1] 104

In [22]:
table(unmatched_event_desc$file.panel)


PB1 PM1 PS1 PT1 
 26  26  26  26 

In [23]:
selected_event_desc <- do.call(rbind, list(matched_event_desc, batch_event_desc, unmatched_event_desc))

In [24]:
selected_event_csv <- paste0("output/human_immune_health_atlas_flow-fcs_file-meta_", Sys.Date(), ".csv")
write.csv(
    selected_event_desc,
    selected_event_csv,
    row.names = FALSE,
    quote = FALSE
)

In [25]:
table(selected_event_desc$file.batchID, selected_event_desc$file.panel)

      
       PB1 PM1 PS1 PT1
  B007   1   1   1   1
  B010   1   1   1   1
  B014   3   3   3   3
  B015   2   2   2   2
  B022   3   3   3   3
  B036   2   2   2   2
  B039   4   4   4   4
  B040   9   9   9   9
  B041   6   6   6   6
  B043   2   2   2   2
  B045   3   3   3   3
  B046   6   6   6   6
  B053   5   5   5   5
  B054   2   2   2   2
  B055   1   1   1   1
  B056   4   4   4   4
  B057   2   2   2   2
  B060   3   3   3   3
  B063   2   2   2   2
  B064   1   1   1   1
  B067   1   1   1   1
  B072   5   5   5   5
  B074   1   1   1   1
  B077   3   3   3   3
  B079   3   3   3   3
  B080   2   2   2   2
  B082   1   1   1   1
  B084   2   2   2   2
  B085   1   1   1   1
  B091   4   4   4   4
  B094   4   4   4   4
  B096   4   4   4   4
  B132   1   1   1   1
  B138   3   3   3   3
  B142   1   1   1   1
  B145   2   2   2   2
  B151   8   8   8   8

Let's split these by panel and retrieve the data.

In [26]:
panel_events <- split(selected_event_desc, selected_event_desc$file.panel)

In [27]:
events_files <- map(
    panel_events,
    function(df) {
        cacheFiles(as.list(df[["file.id"]]))
    }
)

[1] "downloading fileID 1cbb0b99-a146-4445-965b-503e6754edbd"
[1] "downloading fileID 474f76a9-6fbd-418f-9ca6-8305f9efb158"
[1] "downloading fileID 20ae691b-ff34-4d27-b64b-9a9b238de4cf"
[1] "downloading fileID 0967cc69-a8ce-4e19-9a44-60d8ecb9a79c"
[1] "downloading fileID 30d256ff-c13d-4924-935c-021790b3a831"
[1] "downloading fileID c2dc0116-96e3-4b1a-8693-4c8a09e785a5"
[1] "downloading fileID a8258749-3b7a-4a62-bd3d-0da504af7b0e"
[1] "downloading fileID 4a088235-33c4-4011-bc3a-95c72b357c6e"
[1] "downloading fileID e903ad8a-5b57-4048-9457-d87d3c741050"
[1] "downloading fileID 710ad404-4b87-45a4-8d0b-f4cfcfbc5136"
[1] "downloading fileID 06f1a4d4-bd63-4444-a25a-71112043d63a"
[1] "downloading fileID 2cb55ad3-552c-42b7-a9f6-4fc81c5377e9"
[1] "downloading fileID f10446d2-8a4a-496c-9561-146f94dfc100"
[1] "downloading fileID dd362be5-045c-40d3-a706-082c88a89f91"
[1] "downloading fileID a10fa302-33d3-4300-aeb5-75bf2f853d7b"
[1] "downloading fileID 0e19e03c-bfbe-4a8d-961d-1ce99d4adc57"
[1] "dow

In [28]:
events_files <- map(
    events_files,
    function(files) {
        paste0("/home/workspace/", files)
    }
)

In [29]:
events_files[[1]][1:3]

[1] "/home/workspace/input/1918706177/cohorts/1cbb0b99-a146-4445-965b-503e6754edbd/AIFI-2022-10-03T23:12:39.605096139Z/B007_PB1_PB00041-01_QC.fcs"
[2] "/home/workspace/input/1918706177/cohorts/474f76a9-6fbd-418f-9ca6-8305f9efb158/AIFI-2022-10-10T18:14:32.90838824Z/B010_PB1_PB00166-01_QC.fcs" 
[3] "/home/workspace/input/1918706177/cohorts/20ae691b-ff34-4d27-b64b-9a9b238de4cf/AIFI-2022-08-11T22:49:04.73057516Z/B039_PB1_PB00334-01_QC.fcs"

## Build tar file for each panel

First, copy and rename files

In [30]:
cohort_names <- c(
    "BR1" = "Young-Adult",
    "BR2" = "Older-Adult",
    "UP1" = "Pediatric"
)

In [31]:
copy_flow_file <- function(
    in_file,
    sample_meta,
    cohort_names
) {
    in_uuid <- sub("/home/workspace/input/[0-9]+/[^/]+/(.+)/AIFI.+", "\\1", in_file)
    sample_idx <- which(sample_meta$file.id == in_uuid)

    cohort <- cohort_names[sample_meta$cohort.cohortGuid[sample_idx]]
    subject <- sample_meta$subject.subjectGuid[sample_idx]
    visit <- sample_meta$sample.visitName[sample_idx]
    visit <- gsub(" ", "-", visit)
    kit <- sample_meta$sample.sampleKitGuid[sample_idx]
    panel <- sample_meta$file.panel[sample_idx]

    out_file <- paste0(cohort, "_", subject, "_", visit, "_", kit, "_", panel, ".fcs")
    out_path <- paste0("output/human_immune_health_atlas_flow-fcs_panel-", panel , "/", out_file)

    file.copy(in_file, out_path)
}

In [32]:
panel_dirs <- map(
    c("PB1", "PM1", "PS1", "PT1"),
    function(panel) {
        paste0("output/human_immune_health_atlas_flow-fcs_panel-", panel)
    }
)
names(panel_dirs) <- c("PB1", "PM1", "PS1", "PT1")

In [33]:
walk2(
    events_files, names(events_files),
    function(fileset, panel) {
        
        panel_dir <- panel_dirs[[panel]]
        if(!dir.exists(panel_dir)) {
            dir.create(panel_dir)
            panel_file <- panel_files[[panel]]
            file.copy(panel_file, paste0(panel_dir, "/", basename(panel_file)))
        }
        
        walk(
            fileset,
            copy_flow_file,
            sample_meta = selected_event_desc,
            cohort_names = cohort_names
        )
    }
)

Then, generate .tar bundles. Compression (.tar.gz) takes a lot of time, but has minimal effects on the output file size, as most of the data is already compressed in the .fcs files.

In [34]:
panel_tars <- map(
    panel_dirs,
    function(d) {
        paste0(d, ".tar")
    }
)

In [35]:
walk2(
    panel_dirs, panel_tars,
    function(dir, tar) {
        system(paste("tar -cf", tar, dir))
    }
)

## Upload data to HISE

In [36]:
study_space_uuid <- "64097865-486d-43b3-8f94-74994e0a72e0"
title <- paste("Imm. Health Atlas Flow .fcs files", Sys.Date())

In [37]:
search_id <- ids::proquint(n_words = 3)

In [43]:
search_id

[1] "dudan-dimuz-porih"

In [38]:
in_list <- as.list(
    selected_event_desc[["file.id"]]
)

In [39]:
out_list <- c(panel_tars, list(selected_event_csv))

In [40]:
out_list

$PB1
[1] "output/human_immune_health_atlas_flow-fcs_panel-PB1.tar"

$PM1
[1] "output/human_immune_health_atlas_flow-fcs_panel-PM1.tar"

$PS1
[1] "output/human_immune_health_atlas_flow-fcs_panel-PS1.tar"

$PT1
[1] "output/human_immune_health_atlas_flow-fcs_panel-PT1.tar"

[[5]]
[1] "output/human_immune_health_atlas_flow-fcs_file-meta_2024-10-29.csv"

In [41]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = in_list,
    store = "project",
    destination = search_id
)

$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "75ef4fa8-fa34-4eef-af91-9e19b3994916"

$ProcessId
[1] "5575e9e0-6115-4e4a-9f52-5fb023e0bce4"

$WorkflowId
[1] "f526ff55-fa34-48fa-8288-3b6562d8cf77"

$FileIds
$FileIds[[1]]
[1] "7fae0c52-fe3c-494c-a4ad-4235a1eb3b41"

$FileIds[[2]]
[1] "87b8dd66-14af-4243-961e-e1eace33202a"

$FileIds[[3]]
[1] "c5fe34ae-4251-4dc9-912d-2dbb85afe418"

$FileIds[[4]]
[1] "81929c4f-d3f4-415d-9761-8e6f40ed617f"

$FileIds[[5]]
[1] "2cc9a15f-28f3-43ab-b2e2-5e7c0d0490a2"

In [42]:
sessionInfo()

R version 4.1.3 (2022-03-10)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 24.04 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/r_flow/lib/libopenblasp-r0.3.20.so

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] purrr_0.3.4 dplyr_1.0.8 hise_2.16.0

loaded via a namespace (and not attached):
 [1] Rcpp_1.0.8.3     plyr_1.8.7       pillar_1.7.0     compiler_4.1.3  
 [5] base64enc_0.1-3  bitops_1.0-7     tools_4.1.3      digest_0.6.29   
 [9] uuid_1.1-0       jsonlite_1.8.0   evaluate_0.15    lifecycle_1.0.1 
[13] tibble_3.1.6     pkgconfig_2.0.3  rlang_1.0.2      IRdisplay_1.1   
[17] cli_3.2.0        DBI